In [0]:
use catalog identifier(:catalog)


In [0]:

create or replace table gold.dim_geography AS
SELECT n.nationkey AS nation_key,
       n.name AS nation_name,
       n.regionkey AS region_key,
       r.name AS region_name
FROM silver.nation n
LEFT JOIN silver.region r ON r.regionkey = n.regionkey

num_affected_rows,num_inserted_rows


In [0]:
create or replace table gold.dim_passenger AS
SELECT p.passengerid AS passenger_id,
       p.name AS passenger_name,
       p.address AS address,
       p.phone AS phone,
       p.loyaltypoints AS loyalty_points,
       p.travelpurpose AS travel_purpose,
       p.nationkey AS nation_key,
       p.comment AS comment
FROM silver.passenger p

num_affected_rows,num_inserted_rows


In [0]:

create or replace table gold.dim_route AS
SELECT r.routeid AS route_id,
       r.name AS route_name,
       r.aircraft AS aircraft,
       r.airline_brand AS airline_brand,
       r.routetype AS route_type,
       r.capacity AS capacity,
       r.cabinconfig AS cabin_config,
       r.baseprice AS base_price,
       r.comment AS comment
FROM silver.route r

num_affected_rows,num_inserted_rows


In [0]:

create or replace table gold.dim_airline AS
SELECT a.airlineid AS airline_id,
       a.name AS airline_name,
       a.address AS address,
       a.phone AS phone,
       a.rating AS rating,
       a.nationkey AS nation_key,
       a.comment AS comment
FROM silver.airline a

num_affected_rows,num_inserted_rows


In [0]:
create or replace table gold.fact_bookings AS
SELECT b.bookingid AS booking_id,
       fb.linenumber AS line_number,
       b.passengerid AS passenger_id,
       fb.routeid AS route_id,
       fb.airlineid AS airline_id,
       p.nationkey AS passenger_nation_key,
       al.nationkey AS airline_nation_key,
       fb.seats AS seat_count,
       fb.ticketprice AS ticket_price,
       fb.discount AS discount_rate,
       fb.tax AS tax_amount,
       fb.refundflag AS refund_flag,
       fb.segmentstatus AS segment_status,
       fb.cabinclass AS cabin_class,
       fb.departuredate AS departure_date,
       fb.arrivaldate AS arrival_date,
       fb.commitdate AS commit_date,
       b.bookingdate AS booking_date,
       b.flightdate AS flight_date,
       b.status AS booking_status,
       b.agent AS agent,
       b.bookingpriority AS booking_priority,
       fb.ticketprice AS gross_revenue,
       fb.ticketprice * (1 - fb.discount) AS net_revenue
FROM silver.flight_booking fb
LEFT JOIN silver.booking b   ON b.bookingid = fb.bookingid
LEFT JOIN silver.passenger p ON p.passengerid = b.passengerid
LEFT JOIN silver.airline al  ON al.airlineid = fb.airlineid

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE VIEW gold.vw_annual_booking_kpis AS
WITH revenue AS (
  SELECT year(booking_date) AS booking_year,
         SUM(net_revenue) AS net_revenue,
         count(distinct booking_id) as distinct_counts
  FROM gold.fact_bookings
  GROUP BY year(booking_date)
),
bookers AS (
  SELECT year(bookingdate) AS booking_year,
         COUNT(DISTINCT bookingid) AS distinct_bookings,
         COUNT(DISTINCT passengerid) AS active_passengers
  FROM silver.booking
  GROUP BY year(bookingdate)
)
SELECT b.booking_year,
       ROUND(COALESCE(r.net_revenue, 0), 2) AS net_revenue,
       COALESCE(r.distinct_counts, 0) AS distinct_bookings,
       b.active_passengers,
       ROUND(COALESCE(r.net_revenue, 0) / NULLIF(COALESCE(r.distinct_counts, 0), 0), 2) AS avg_booking_value
FROM bookers b
LEFT JOIN revenue r ON r.booking_year = b.booking_year

In [0]:
create or replace table gold.gold_silver_reconciliation AS
SELECT metric, gold_value, silver_value, ROUND(gold_value - silver_value, 2) AS difference,
       CASE WHEN abs(gold_value - silver_value) < 0.005 THEN 'PASS' ELSE 'FAIL' END AS status
FROM (
  SELECT 'fact_row_count' AS metric,
         (SELECT count(*) FROM gold.fact_bookings) AS gold_value,
         (SELECT count(*) FROM silver.flight_booking) AS silver_value
  UNION ALL
  SELECT 'net_revenue_total',
         (SELECT ROUND(sum(net_revenue), 2) FROM gold.fact_bookings),
         (SELECT ROUND(sum(ticketprice * (1 - discount)), 2) FROM silver.flight_booking)
  UNION ALL
  SELECT 'gross_revenue_total',
         (SELECT ROUND(sum(gross_revenue), 2) FROM gold.fact_bookings),
         (SELECT ROUND(sum(ticketprice), 2) FROM silver.flight_booking)
  UNION ALL
  SELECT 'distinct_bookings',
         (SELECT count(DISTINCT booking_id) FROM gold.fact_bookings),
         (SELECT count(DISTINCT bookingid) FROM silver.flight_booking)
  UNION ALL
  SELECT 'active_passengers',
         (SELECT count(DISTINCT passenger_id) FROM gold.fact_bookings),
         (SELECT count(DISTINCT bk.passengerid) FROM silver.flight_booking fb
            JOIN silver.booking bk ON bk.bookingid = fb.bookingid)
)

num_affected_rows,num_inserted_rows


In [0]:
create or replace table gold.dim_key_validation AS
SELECT 'dim_geography' AS dimension, 'nation_key' AS key_column,
       count(*) AS row_count, count(DISTINCT nation_key) AS distinct_keys,
       CASE WHEN count(*) = count(DISTINCT nation_key) THEN 'PASS' ELSE 'FAIL' END AS uniqueness_status
FROM gold.dim_geography
UNION ALL
SELECT 'dim_passenger', 'passenger_id', count(*), count(DISTINCT passenger_id),
       CASE WHEN count(*) = count(DISTINCT passenger_id) THEN 'PASS' ELSE 'FAIL' END
FROM gold.dim_passenger
UNION ALL
SELECT 'dim_route', 'route_id', count(*), count(DISTINCT route_id),
       CASE WHEN count(*) = count(DISTINCT route_id) THEN 'PASS' ELSE 'FAIL' END
FROM gold.dim_route
UNION ALL
SELECT 'dim_airline', 'airline_id', count(*), count(DISTINCT airline_id),
       CASE WHEN count(*) = count(DISTINCT airline_id) THEN 'PASS' ELSE 'FAIL' END
FROM gold.dim_airline

num_affected_rows,num_inserted_rows
